# 04 — Advanced Feature Engineering

164 fitur kanonik (`src/features/engineer.py`) sudah mencakup HCP, panjang
suit, stopper, LTC, fit, dsb. Notebook ini menambah fitur turunan yang
**belum ada**, berbasis konsep evaluasi tangan bridge yang lebih canggih:

| Fitur baru | Level | Rasional |
|---|---|---|
| `{seat}_quick_tricks` | per-hand | Quick Tricks (AK=2, AQ=1.5, A=1, KQ=1, Kx=0.5) — ukuran kekuatan "pasti menang" yang beda dari HCP mentah |
| `{seat}_total_points` | per-hand | HCP + shortness points (void=3/singleton=2/doubleton=1) — valuasi tangan gaya "total points", pelengkap HCP murni |
| `{seat}_suit_quality_best` | per-hand | Jumlah honor (A/K/Q/J/T) di suit terpanjang — kualitas suit untuk preempt/lead, bukan cuma panjangnya |
| `{prefix}quick_tricks`, `{prefix}total_points` | partnership | Gabungan versi partnership dari dua fitur di atas |
| `{prefix}second_fit` | partnership | Panjang fit terbaik KEDUA (pelengkap `{prefix}best_fit` yang sudah ada) — relevan untuk tangan dua-suited |
| `{prefix}lott_level` | partnership | `best_fit - 6`, estimasi level ala Law of Total Tricks |
| `opener_hcp`, `opener_balanced` | deal | HCP & keseimbangan tangan pembuka lelang pertama (bukan dealer — bisa beda kalau dealer pass) |
| `preemptive_opening` | deal | 1 jika opening level >= 2 DAN HCP pembuka < 10 — sinyal lelang barrage/weak two |

**Anti-leakage**: semua fitur baru dihitung dari kartu tangan atau bid
PEMBUKA (bid pertama non-pass) — tidak ada yang memakai bid TERAKHIR atau
level kontrak final, supaya tidak membocorkan target secara langsung
(prinsip yang sama dipakai `opening_level`/`opening_strain_*` yang sudah
ada di 164 fitur kanonik).

Fitur baru dihitung dengan parsing ulang `data/raw/` (tidak mengubah
`src/features/engineer.py` ataupun `data/processed/` kanonik), lalu
di-join ke split train/val/test yang sudah ada lewat kunci identitas
`(_source_file, _room, _board_number)` — sehingga split 70/15/15 dan
seed 42 tetap persis sama.


In [1]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.parser import LINParser
from src.features.engineer import (
    extract_features, SUITS, SEATS,
    hand_hcp_total, hand_is_balanced, hand_void_count,
    hand_singleton_count, hand_doubleton_count,
)
from src.preprocessing.dataset_builder import load_splits
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-09" / "outputs" / "advanced_feature_engineering"
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Fungsi fitur baru

In [2]:
SEATS_ORDER = ["N", "E", "S", "W"]


def suit_quick_tricks(cards: list[str]) -> float:
    s = set(cards)
    if "A" in s and "K" in s:
        return 2.0
    if "A" in s and "Q" in s:
        return 1.5
    if "A" in s:
        return 1.0
    if "K" in s and "Q" in s:
        return 1.0
    if "K" in s and len(cards) >= 2:
        return 0.5
    return 0.0


def hand_quick_tricks(hand) -> float:
    return sum(suit_quick_tricks(cards) for cards in hand.as_dict().values())


def hand_total_points(hand) -> int:
    hcp = hand_hcp_total(hand)
    shortness = (
        3 * hand_void_count(hand)
        + 2 * hand_singleton_count(hand)
        + 1 * hand_doubleton_count(hand)
    )
    return hcp + shortness


def suit_quality(cards: list[str]) -> int:
    return sum(1 for c in cards if c in ("A", "K", "Q", "J", "T"))


def hand_suit_quality_best(hand) -> int:
    lengths = {s: len(cards) for s, cards in hand.as_dict().items()}
    best_suit = max(lengths, key=lambda s: (lengths[s], SUITS.index(s)))
    return suit_quality(hand.as_dict()[best_suit])


def second_best_fit(h1, h2) -> int:
    combined = {
        s: len(h1.as_dict()[s]) + len(h2.as_dict()[s])
        for s in SUITS
    }
    ordered = sorted(combined.values(), reverse=True)
    return ordered[1] if len(ordered) > 1 else 0


def get_opener_seat(board) -> str | None:
    if not board.dealer or board.dealer not in SEATS_ORDER:
        return None
    dealer_idx = SEATS_ORDER.index(board.dealer)
    for i, bid in enumerate(board.auction):
        b = bid.upper().strip()
        if b not in ("P", "PASS", "AP"):
            return SEATS_ORDER[(dealer_idx + i) % 4]
    return None


def extract_advanced_features(board, base_row: dict) -> dict:
    f = {}
    for seat in SEATS:
        hand = board.hands[seat]
        f[f"{seat}_quick_tricks"] = hand_quick_tricks(hand)
        f[f"{seat}_total_points"] = hand_total_points(hand)
        f[f"{seat}_suit_quality_best"] = hand_suit_quality_best(hand)

    for prefix, (s1, s2) in {"ns_": ("N", "S"), "ew_": ("E", "W")}.items():
        h1, h2 = board.hands[s1], board.hands[s2]
        f[f"{prefix}quick_tricks"] = hand_quick_tricks(h1) + hand_quick_tricks(h2)
        f[f"{prefix}total_points"] = hand_total_points(h1) + hand_total_points(h2)
        f[f"{prefix}second_fit"] = second_best_fit(h1, h2)
        best_fit_key = f"{prefix}best_fit"
        f[f"{prefix}lott_level"] = base_row[best_fit_key] - 6

    opener_seat = get_opener_seat(board)
    if opener_seat is not None:
        opener_hand = board.hands[opener_seat]
        f["opener_hcp"] = hand_hcp_total(opener_hand)
        f["opener_balanced"] = int(hand_is_balanced(opener_hand))
    else:
        f["opener_hcp"] = 0
        f["opener_balanced"] = 0

    f["preemptive_opening"] = int(
        base_row.get("opening_level", 0) >= 2 and f["opener_hcp"] < 10
    )

    # Identity for joining back onto existing splits
    f["_source_file"] = board.source_file
    f["_room"] = board.room
    f["_board_number"] = board.board_number
    return f


## 2. Parse ulang `data/raw/` dan hitung fitur baru

In [3]:
t0 = time.time()
parser = LINParser()
boards = parser.parse_directory(PROJECT_ROOT / "data" / "raw")
print(f"Parsed {len(boards)} boards in {time.time() - t0:.1f}s")

new_rows = []
for b in boards:
    base_row = extract_features(b)
    if base_row is None:
        continue
    new_rows.append(extract_advanced_features(b, base_row))

df_new = pd.DataFrame(new_rows)
new_feature_cols = [c for c in df_new.columns if not c.startswith("_")]
print(f"New feature columns ({len(new_feature_cols)}): {new_feature_cols}")
df_new.head()


Parsed 11236 boards in 8.5s


New feature columns (23): ['N_quick_tricks', 'N_total_points', 'N_suit_quality_best', 'E_quick_tricks', 'E_total_points', 'E_suit_quality_best', 'S_quick_tricks', 'S_total_points', 'S_suit_quality_best', 'W_quick_tricks', 'W_total_points', 'W_suit_quality_best', 'ns_quick_tricks', 'ns_total_points', 'ns_second_fit', 'ns_lott_level', 'ew_quick_tricks', 'ew_total_points', 'ew_second_fit', 'ew_lott_level', 'opener_hcp', 'opener_balanced', 'preemptive_opening']


,N_quick_tricks,N_total_points,N_suit_quality_best,E_quick_tricks,E_total_points,E_suit_quality_best,S_quick_tricks,S_total_points,S_suit_quality_best,W_quick_tricks,...,ew_quick_tricks,ew_total_points,ew_second_fit,ew_lott_level,opener_hcp,opener_balanced,preemptive_opening,_source_file,_room,_board_number
0,0.5,3,2,1.5,11,2,1.5,12,3,4.0,...,5.5,28,7,1,11,1,0,85168.lin,open,13
1,0.5,3,2,1.5,11,2,1.5,12,3,4.0,...,5.5,28,7,1,11,1,0,85168.lin,closed,13
2,0.5,9,2,2.5,15,2,3.0,15,2,0.5,...,3.0,25,7,4,13,0,0,85168.lin,open,14
3,0.5,9,2,2.5,15,2,3.0,15,2,0.5,...,3.0,25,7,4,13,0,0,85168.lin,closed,14
4,1.5,10,2,3.0,16,2,1.0,6,2,1.0,...,4.0,26,7,2,15,1,0,85168.lin,open,15


## 3. Join ke split train/val/test yang sudah ada (kunci identitas, split & seed tidak berubah)

In [4]:
df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed")

JOIN_KEYS = ["_source_file", "_room", "_board_number"]


def augment(df_split: pd.DataFrame) -> pd.DataFrame:
    before = len(df_split)
    merged = df_split.merge(df_new, on=JOIN_KEYS, how="left", validate="one_to_one")
    assert len(merged) == before, "Join changed row count — identity key isn't unique enough"
    assert merged[new_feature_cols].isnull().sum().sum() == 0, "Some rows didn't find a match"
    return merged


df_train_aug = augment(df_train)
df_val_aug = augment(df_val)
df_test_aug = augment(df_test)

print("Train aug shape:", df_train_aug.shape, "| Val aug shape:", df_val_aug.shape)


Train aug shape: (7157, 197) | Val aug shape: (1533, 197)


## 4. Bandingkan XGBoost: 164 fitur (baseline) vs 164+baru (augmented)

In [5]:
BASE_XGB_PARAMS = dict(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, eval_metric="mlogloss", verbosity=0,
)

X_train_base = df_train_aug[feature_cols].values
X_val_base = df_val_aug[feature_cols].values
y_train = df_train_aug["label"].values
y_val = df_val_aug["label"].values

all_feature_cols = feature_cols + new_feature_cols
X_train_aug = df_train_aug[all_feature_cols].values
X_val_aug = df_val_aug[all_feature_cols].values

t0 = time.time()
clf_base = XGBClassifier(**BASE_XGB_PARAMS)
clf_base.fit(X_train_base, y_train)
res_base = evaluate(y_val, clf_base.predict(X_val_base), clf_base.predict_proba(X_val_base),
                     le, model_name="XGBoost (164 fitur)")
print(f"Baseline fit: {time.time() - t0:.1f}s")
print_summary(res_base)

t0 = time.time()
clf_aug = XGBClassifier(**BASE_XGB_PARAMS)
clf_aug.fit(X_train_aug, y_train)
res_aug = evaluate(y_val, clf_aug.predict(X_val_aug), clf_aug.predict_proba(X_val_aug),
                    le, model_name=f"XGBoost (164+{len(new_feature_cols)} fitur)")
print(f"Augmented fit: {time.time() - t0:.1f}s")
print_summary(res_aug)


Baseline fit: 30.7s

  XGBoost (164 fitur)
  Accuracy          : 0.5127
  Precision (macro) : 0.3335
  Recall (macro)    : 0.2515
  F1 (macro)        : 0.2685
  F1 (weighted)     : 0.4685
  Top-3 Accuracy    : 0.7508
  Top-5 Accuracy    : 0.8480


Augmented fit: 37.1s

  XGBoost (164+23 fitur)
  Accuracy          : 0.5121
  Precision (macro) : 0.3481
  Recall (macro)    : 0.2590
  F1 (macro)        : 0.2795
  F1 (weighted)     : 0.4708
  Top-3 Accuracy    : 0.7586
  Top-5 Accuracy    : 0.8421


## 5. Ranking pentingnya fitur baru (feature importance XGBoost)

In [6]:
importances = pd.Series(clf_aug.feature_importances_, index=all_feature_cols).sort_values(ascending=False)
importances_rank = pd.DataFrame({
    "feature": importances.index,
    "importance": importances.values,
    "rank": range(1, len(importances) + 1),
    "is_new": [f in new_feature_cols for f in importances.index],
})
print("Top 20 fitur (baru ditandai *):")
for _, row in importances_rank.head(20).iterrows():
    marker = " *NEW*" if row["is_new"] else ""
    print(f"  #{row['rank']:3d}  {row['feature']:28s} {row['importance']:.4f}{marker}")

print()
print("Semua fitur baru dan rankingnya:")
print(importances_rank[importances_rank["is_new"]].to_string(index=False))


Top 20 fitur (baru ditandai *):
  #  1  ns_has_fit_S                 0.0239
  #  2  ew_has_fit_S                 0.0198
  #  3  ns_fit_S                     0.0173
  #  4  ns_fit_H                     0.0167
  #  5  ew_fit_S                     0.0165
  #  6  ew_hcp                       0.0164
  #  7  ns_hcp                       0.0162
  #  8  ew_total_points              0.0162 *NEW*
  #  9  ew_has_fit_H                 0.0150
  # 10  hcp_ns_advantage             0.0149
  # 11  opening_strain_PASS          0.0148
  # 12  ns_total_points              0.0146 *NEW*
  # 13  ns_has_fit_H                 0.0146
  # 14  ew_fit_H                     0.0128
  # 15  auction_competitive          0.0107
  # 16  ew_best_fit                  0.0101
  # 17  ns_lott_level                0.0098 *NEW*
  # 18  ns_both_balanced             0.0095
  # 19  auction_has_double           0.0092
  # 20  ns_best_suit_D               0.0086

Semua fitur baru dan rankingnya:
            feature  importance  ran

In [7]:
comparison = compare_models([res_base, res_aug])
comparison.to_csv(OUT_DIR / "val_comparison.csv")
importances_rank.to_csv(OUT_DIR / "feature_importance_ranking.csv", index=False)

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "new_feature_cols": new_feature_cols,
    "n_new_features": len(new_feature_cols),
    "results": {r["model"]: {k: r[k] for k in
        ["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]}
        for r in [res_base, res_aug]},
    "new_feature_ranks": importances_rank[importances_rank["is_new"]].to_dict("records"),
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"Saved: {OUT_DIR / 'val_comparison.csv'}")
print(f"Saved: {OUT_DIR / 'feature_importance_ranking.csv'}")
print(f"Saved: {OUT_DIR / 'summary.json'}")


Saved: D:\Bridge-Prediction\experiments\2026-07-09\outputs\advanced_feature_engineering\val_comparison.csv
Saved: D:\Bridge-Prediction\experiments\2026-07-09\outputs\advanced_feature_engineering\feature_importance_ranking.csv
Saved: D:\Bridge-Prediction\experiments\2026-07-09\outputs\advanced_feature_engineering\summary.json


## 6. Kesimpulan

**Diperbarui 2026-07-10 setelah perbaikan split group-aware** — angka di
bawah ini MENGGANTIKAN kesimpulan sebelumnya, dan kesimpulannya BERBALIK.

Hasil val set (`experiments/2026-07-09/outputs/advanced_feature_engineering/val_comparison.csv`):

| Model | Accuracy | F1 macro | F1 weighted | Top-3 | Top-5 |
|---|---|---|---|---|---|
| XGBoost (164 fitur) | 51.3% | 0.269 | 0.469 | 75.1% | **84.8%** |
| XGBoost (164+23 fitur) | 51.2% | **0.280** | **0.471** | **75.9%** | 84.2% |

- **Kesimpulan berbalik dari versi sebelumnya.** Dengan split lama
  (bocor), 23 fitur baru terlihat SEDIKIT MERUGIKAN di semua metrik.
  Dengan split yang diperbaiki, fitur baru memberi **perbaikan kecil
  tapi konsisten** di F1 macro (+0.011) dan F1 weighted (+0.002) dengan
  accuracy nyaris sama (-0.1pp). Artinya kesimpulan awal ("164 fitur
  kanonik sudah cukup, fitur baru redundan") sebagian adalah artefak
  dari split yang bocor, bukan temuan murni tentang fiturnya.
- **Feature importance tetap mengonfirmasi pola yang sama**:
  `ns/ew_total_points` masih tinggi (rank 8 & 12, naik dari rank 10/12
  sebelumnya), `ns/ew_lott_level` juga signifikan (rank 17 & 23). Fitur
  ini memang kombinasi dari fitur lama (HCP+shortness,
  best_fit-6) tapi ternyata kombinasi eksplisit ini MEMBANTU model
  tree-based sedikit, bukan sepenuhnya redundan seperti dugaan awal.
- **Perbaikannya tetap kecil** (F1 macro +0.011, F1 weighted +0.002) —
  tidak mengubah rekomendasi praktis secara drastis, tapi cukup untuk
  membalik kesimpulan dari "jangan tambahkan" menjadi "boleh
  dipertimbangkan, dengan ekspektasi realistis (gain marginal, bukan
  lompatan)".
- **Rekomendasi diperbarui**: fitur baru (khususnya `total_points` dan
  `lott_level` per partnership — yang paling tinggi rank-nya) layak
  dipertimbangkan masuk ke 164 fitur kanonik jika validasi test set
  mengonfirmasi perbaikan ini konsisten, TAPI jangan buru-buru — gain
  yang terukur kecil dan bisa jadi noise di dalam val set ~1.500 baris.
